In [ ]:
%%writefile utils.py

import torch
#FUNÇÃO IoU
def iou(box1, box2):
  # Coordenadas da intersecção
  y1 = torch.max(box1[0], box2[0])
  x1 = torch.max(box1[1], box2[1])
  y2 = torch.min(box1[2], box2[2])
  x2 = torch.min(box1[3], box2[3])

  intersection = (x2 - x1).clamp(0) * (y2 - y1).clamp(0)

  area_1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
  area_2 = (box2[2] - box2[0]) * (box2[3] - box2[1])

  union = area_1 + area_2 - intersection

  if union == 0:
    return 0.0

  return intersection / union
#FUNÇÃO NON-MAX SUPPRESSION
def manual_nms(boxes, scores, iou_threshold = 0.5):
  # Ordenar as caixas pela pontuação em ordem decrescente
  order = scores.argsort(descending=True)
  boxes = boxes[order]
  scores = scores[order]

  selected_boxes = []

  while len(boxes) > 0:
    # Selecionar a caixa com a maior pontuação
    best_box = boxes[0]
    selected_boxes.append(best_box)

    # Remover a melhor caixa e sua pontuação da lista atual
    boxes = boxes[1:]
    scores = scores[1:]

    if len(boxes) == 0:
      break

    # --- Cálculo vetorizado do IoU ---
    # Coordenadas da intersecção para best_box vs. todas as remaining_boxes
    y1_inter = torch.max(best_box[0], boxes[:, 0])
    x1_inter = torch.max(best_box[1], boxes[:, 1])
    y2_inter = torch.min(best_box[2], boxes[:, 2])
    x2_inter = torch.min(best_box[3], boxes[:, 3])

    # Área da intersecção
    intersection_area = (x2_inter - x1_inter).clamp(min=0) * (y2_inter - y1_inter).clamp(min=0)

    # Área da best_box (escalar)
    area_best_box = (best_box[2] - best_box[0]) * (best_box[3] - best_box[1])

    # Área das remaining_boxes (vetor)
    area_remaining_boxes = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])

    # União (vetor)
    union_area = area_best_box + area_remaining_boxes - intersection_area

    # Calcular IoUs, tratando a divisão por zero
    ious = torch.where(union_area == 0, torch.tensor(0.0, device=union_area.device, dtype=union_area.dtype), intersection_area / union_area)
    # --- Fim do cálculo vetorizado do IoU ---

    # Manter apenas as caixas cujo IoU com a melhor caixa é menor ou igual ao limiar
    keep_indices = ious <= iou_threshold

    boxes = boxes[keep_indices]
    scores = scores[keep_indices]

  # Retorna as caixas selecionadas convertidas para int
  return torch.stack(selected_boxes).int() if selected_boxes else torch.empty((0, 4), device=boxes.device, dtype=torch.int)

In [ ]:
import shutil
import os

# Defina o caminho no seu Google Drive onde você quer salvar o arquivo
drive_path = '/content/drive/MyDrive/Colab_Utils'

# Crie a pasta se ela não existir
os.makedirs(drive_path, exist_ok=True)

# Copie o arquivo utils.py para o Google Drive
shutil.copy('utils.py', os.path.join(drive_path, 'utils.py'))

print(f"'utils.py' salvo em: {os.path.join(drive_path, 'utils.py')}")